In [ ]:
!pip install umap-learn

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express
import seaborn
import sklearn
import sklearn.preprocessing
import sklearn.decomposition
import umap

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-wine.git
!ls dataset-wine/

## Chargement des données

Chargez `dataset-wine/wineQualityReds.csv` dans un Dataframe `df` et affichez les 10 premiers exemples

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pandas.read_csv("dataset-wine/wineQualityReds.csv").iloc[:, 1:]
df.head(10)
df.shape

## Extraction des features

Afin de pouvoir utiliser sklearn, mettez, sans la qualité (considérée comme la target), les features dans un tableau numpy `X`

In [ ]:
# Votre code ici

### Solution

In [ ]:
X = df.drop(columns="quality").values
print("Forme des données :", X.shape)

## Normalisation

Utilisez [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) afin de normaliser vos données de telle sorte que, pour chaque feature, la moyenne soit à 0 et l'écart-type à 1.

In [ ]:
# Votre code ici

### Solution

In [ ]:
scaler = sklearn.preprocessing.StandardScaler()
X_scaled = scaler.fit_transform(X)

## PCA

- Effectuez une PCA en ne gardant que les 2 premières composantes que vous stockerez dans `X_pca`. Créez une instance de la classe [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html?highlight=pca#sklearn.decomposition.PCA) de scikit-learn puis utilisez la méthode [`fit_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html?highlight=pca#sklearn.decomposition.PCA.fit_transform).

- Affichez la quantité de variance conservée par chaque composante. Pour cela vous pouvez utiliser l'attribut `explained_variance_ratio_` qui est calculé après un `fit` ou un `fit_transform`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
pca = sklearn.decomposition.PCA(n_components=2
                                )
X_pca = pca.fit_transform(X_scaled)
# pca.fit(X_scaled)
# X_pca = pca.transform(X_scaled)

print("Variance expliquée :",
      ", ".join(f"{v:.2f}" for v in pca.explained_variance_ratio_))
print("Variance expliquée cumulée :",
      ", ".join(f"{v:.2f}"
      for v
      in numpy.cumsum(pca.explained_variance_ratio_)))

components_df = pandas.DataFrame(pca.components_, columns=df.columns[:-1])
components_df

## Plotting

Affichez `X_pca` avec `matplotlib` et `seaborn`

In [ ]:
# Votre code ici

### Solution

In [ ]:
print("Avec seaborn")
seaborn.jointplot(x=X_pca[:,0] , y=X_pca[:,1])
plt.show()

print("Avec matplotlib")
plt.plot(X_pca[:, 0], X_pca[:, 1], ".")
plt.show()

print("Avec plotly")
plotly.express.scatter(X_pca, x=0, y=1, width=600)

## Outliers
Quelques points semblent se démarquer du lot.

Identifiez leur indice afin de les afficher depuis `df`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Celui qui a la valeur max sur la première dimension
max_pc0 = numpy.argmax(X_pca[:, 0])
print(max_pc0)
print(df.iloc[max_pc0, :])

In [ ]:
# Les 5 en haut
top5_pc1 = numpy.argsort(-X_pca[:, 1])[:5]
df.iloc[top5_pc1, :]

## The Good, the Bad and the Ugly
Commencez par visualiser la répartition des qualités des vins (via un barplot par exemple).

Réutilisez ensuite les informations de qualité pour séparer vos vins en trois groupes :
 * Les mauvais vins (qualité < 5)
 * Les vins moyens (qualité 5 et 6)
 * Les bons vins (qualité >= 7)

Affichez ensuite votre réduction de dimension via un nuage de points en indiquant avec la couleur du point sa qualité.

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Répartition des qualités
seaborn.countplot(x=df['quality'])
plt.show()

In [ ]:
bad_wine_ind  = df['quality'] < 5
mean_wine_ind = (df['quality'] >= 5) & (df['quality'] < 7)
good_wine_ind = df['quality'] >= 7

def affiche(data):
  f, ax = plt.subplots(figsize=(10, 10))
  plt.scatter(data[mean_wine_ind, 0], data[mean_wine_ind, 1], color = "b",
              alpha=0.05)
  plt.scatter(data[bad_wine_ind, 0], data[bad_wine_ind, 1], color = "r")
  plt.scatter(data[good_wine_ind, 0], data[good_wine_ind, 1], color ="g")
  plt.show()

affiche(X_pca)

## Projection non-linéaire

Utilisez le modèle UMAP du paquet umap (qui s'utilise comme PCA) pour calculer une projection non-linéaire. Comparez la à la projection obtenue par PCA.

In [ ]:
# Votre code ici

### Solution

In [ ]:
affiche(umap.UMAP().fit_transform(X_scaled))

In [ ]:
styling = dict(template="seaborn", width=800, height=600)

def show(fig):
  fig.update_traces(marker=dict(line=dict(width=0.1)),
                    selector=dict(mode='markers'))
  fig.show()


X_umap = umap.UMAP(n_components=3).fit_transform(X_scaled)
fig = plotly.express.scatter_3d(pandas.DataFrame(X_umap),
                                x=0,
                                y=1,
                                z=2,
                                size_max=0.1,
                                color=df["quality"],
                                opacity= 0.6,
                                **styling)
fig.update_traces(marker=dict(size=2.5,
                              line=dict(width=1, color=df["quality"])),
                  selector=dict(mode='markers'))
show(fig)